In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

if not os.getenv("FLIGHTAI_ADMIN_TOKEN"):
    print("Warning: FLIGHTAI_ADMIN_TOKEN is not set; price updates will be rejected.")

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Never call set_ticket_price unless the user provides a valid admin token.
"""

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the ticket price for a destination city",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {"type": "string"}
                },
                "required": ["destination_city"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_ticket_price",
            "description": "Set or update the ticket price for a destination city",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {"type": "string"},
                    "price": {"type": "number", "minimum": 0},
                    "admin_token": {
                        "type": "string",
                        "description": "Admin token required to update prices",
                    },
                },
                "required": ["destination_city", "price", "admin_token"],
                "additionalProperties": False
            }
        }
    }
]

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(destination_city):
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {destination_city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
def set_ticket_price(destination_city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',
            (destination_city.lower(), price, price),
        )
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if name == "get_ticket_price":
            result = get_ticket_price(args["destination_city"])
        elif name == "set_ticket_price":
            expected = os.getenv("FLIGHTAI_ADMIN_TOKEN")
            if not expected or args.get("admin_token") != expected:
                result = "Not authorized to update ticket prices."
            else:
                set_ticket_price(args["destination_city"], args["price"])
                result = f"Updated ticket price for {args['destination_city']} to ${args['price']}"
        else:
            result = "Unknown tool"

        responses.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        })
    return responses

In [ ]:
demo = gr.ChatInterface(fn=chat, title="FlightAI")
demo.launch()